In [ ]:
Extraer información de RemoteOK, usando API publica 

In [16]:
# ----------------------------
# 0️⃣ Librerías
# ----------------------------
import requests
import pandas as pd
import os
import re
from datetime import datetime

# ----------------------------
# 1️⃣ Carpeta para guardar CSV
# ----------------------------
BASE_DIR = os.path.dirname(os.path.abspath(""))  # carpeta del Notebook
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# ----------------------------
# 2️⃣ URL API RemoteOK
# ----------------------------
API_URL = "https://remoteok.com/api"

# ----------------------------
# 3️⃣ Niveles y patrones de experiencia
# ----------------------------
xp_levels = ["junior", "mid-level", "mid level", "senior", "lead", "manager"]

# Patrones de inglés
english_levels = [
    "english", "fluent english", "english required",
    "intermediate english", "advanced english"
]

# ----------------------------
# 4️⃣ Funciones auxiliares
# ----------------------------
def detect_years(text):
    """Detecta años de experiencia en el texto."""
    pattern = r"(\d+)[\+\-]?\d*\s+years?"
    match = re.search(pattern, text.lower())
    if match:
        return match.group(0)
    return ""

def detect_english(text):
    """Detecta nivel de inglés en el texto."""
    for level in english_levels:
        if level.lower() in text.lower():
            return level
    return ""

# ----------------------------
# 5️⃣ Descargar datos de RemoteOK
# ----------------------------
response = requests.get(API_URL, headers={"User-Agent": "Mozilla/5.0"})
if response.status_code != 200:
    raise Exception(f"Error al conectar con RemoteOK: {response.status_code}")

data = response.json()
jobs = data[1:]  # ignorar primera fila general

# ----------------------------
# 6️⃣ Procesar todas las vacantes
# ----------------------------
all_jobs = []

for job in jobs:
    combined_text = " ".join([
        str(job.get("position", "")),
        " ".join(job.get("tags", [])),
        str(job.get("company", "")),
        str(job.get("description", "")) if "description" in job else ""
    ])
    
    # Nivel de experiencia
    level = ""
    for xp in xp_levels:
        if xp in combined_text.lower():
            level = xp
            break
    
    # Años de experiencia
    years = detect_years(combined_text)
    
    # Nivel de inglés
    english = detect_english(combined_text)
    
    all_jobs.append({
        "date": job.get("date"),
        "company": job.get("company"),
        "position": job.get("position"),
        "location": job.get("location"),
        "tags": ", ".join(job.get("tags", [])),
        "remote": job.get("remote"),
        "experience_level": level,
        "experience_years": years,
        "english_level": english,
        "url": job.get("url")
    })

# ----------------------------
# 7️⃣ Guardar CSV en data/raw/
# ----------------------------
df = pd.DataFrame(all_jobs)
today = datetime.today().strftime("%Y_%m_%d")
csv_file = os.path.join(RAW_DATA_DIR, f"remoteok_jobs_all_{today}.csv")
df.to_csv(csv_file, index=False)

print(f"✅ {len(df)} vacantes guardadas en {csv_file}")


✅ 96 vacantes guardadas en d:\PROYECTO JOB\Job-Market-Analysis-\data\raw\remoteok_jobs_all_2026_02_03.csv


In [ ]:
Extraer infromacion de Indeed

  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.6 MB 11.8 MB/s eta 0:00:01
   ------------------- -------------------- 4.7/9.6 MB 11.8 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 11.8 MB/s eta 0:00:

In [32]:
# ----------------------------
# 0️⃣ Librerías
# ----------------------------
import os
import time
import random
import re
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# ----------------------------
# 1️⃣ Carpeta salida
# ----------------------------
BASE_DIR = os.path.dirname(os.path.abspath(""))
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# ----------------------------
# 2️⃣ Configuración Selenium
# ----------------------------
options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless")  # evitar para reducir captcha

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# ----------------------------
# 3️⃣ Query (UNA por ejecución)
# ----------------------------
QUERY = "software engineer"
LOCATION = ""

# ----------------------------
# 4️⃣ Funciones auxiliares
# ----------------------------
xp_levels = ["junior", "mid-level", "mid level", "senior", "lead", "manager"]
english_levels = [
    "english",
    "fluent english",
    "english required",
    "intermediate english",
    "advanced english"
]

def detect_years(text):
    pattern = r"(\d+)[\+\-]?\d*\s+years?"
    match = re.search(pattern, text.lower())
    return match.group(0) if match else ""

def detect_english(text):
    for level in english_levels:
        if level.lower() in text.lower():
            return level
    return ""

# ----------------------------
# 5️⃣ Abrir Indeed
# ----------------------------
url = f"https://www.indeed.com/jobs?q={QUERY}&l={LOCATION}"
driver.get(url)

print("⏳ Si aparece captcha, resuélvelo manualmente...")
time.sleep(15)

# ----------------------------
# 6️⃣ Scroll humano
# ----------------------------
last_height = driver.execute_script("return document.body.scrollHeight")

for _ in range(3):
    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
    time.sleep(random.uniform(3, 6))

    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# ----------------------------
# 7️⃣ Extraer vacantes
# ----------------------------
soup = BeautifulSoup(driver.page_source, "html.parser")
jobs_html = soup.find_all("div", class_="job_seen_beacon")

print(f"✅ {len(jobs_html)} vacantes encontradas")

all_jobs = []

for job in jobs_html:
    title_tag = job.find("h2")
    title = title_tag.text.strip() if title_tag else ""

    company_tag = job.find("span", class_="companyName")
    company = company_tag.text.strip() if company_tag else ""

    location_tag = job.find("div", class_="companyLocation")
    location_name = location_tag.text.strip() if location_tag else ""

    link_tag = title_tag.find("a") if title_tag else None
    job_url = "https://www.indeed.com" + link_tag["href"] if link_tag else ""

    remote = "remote" in location_name.lower() if location_name else False

    combined_text = f"{title} {company}"
    level = ""
    for xp in xp_levels:
        if xp in combined_text.lower():
            level = xp
            break

    years = detect_years(combined_text)
    english = detect_english(combined_text)

    date_posted = datetime.today().strftime("%Y-%m-%dT%H:%M:%S+00:00")

    all_jobs.append({
        "date": date_posted,
        "company": company,
        "position": title,
        "location": location_name,
        "tags": QUERY,
        "remote": remote,
        "experience_level": level,
        "experience_years": years,
        "english_level": english,
        "url": job_url
    })

# ----------------------------
# 8️⃣ Guardar CSV
# ----------------------------
driver.quit()

df = pd.DataFrame(all_jobs)
today = datetime.today().strftime("%Y_%m_%d")
csv_file = os.path.join(RAW_DATA_DIR, f"indeed_{QUERY}_{today}.csv")
df.to_csv(csv_file, index=False)

print(f"\n🎉 Datos guardados en:\n{csv_file}")


⏳ Si aparece captcha, resuélvelo manualmente...
✅ 16 vacantes encontradas

🎉 Datos guardados en:
d:\PROYECTO JOB\Job-Market-Analysis-\data\raw\indeed_software engineer_2026_02_09.csv


In [ ]:
Extraer infromacion de Remotive

In [33]:
# ----------------------------
# 0️⃣ Librerías
# ----------------------------
import requests
import pandas as pd
import os
import re
from datetime import datetime

# ----------------------------
# 1️⃣ Carpeta salida
# ----------------------------
BASE_DIR = os.path.dirname(os.path.abspath(""))
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# ----------------------------
# 2️⃣ API Remotive
# ----------------------------
API_URL = "https://remotive.com/api/remote-jobs"

# Queries relacionadas a desarrollo y datos
KEYWORDS = [
    "python", "data", "developer", "engineer",
    "machine learning", "backend", "frontend",
    "full stack", "ai"
]

# ----------------------------
# 3️⃣ Patrones experiencia
# ----------------------------
xp_levels = ["junior", "mid-level", "mid level", "senior", "lead", "manager"]

english_levels = [
    "english",
    "fluent english",
    "english required",
    "intermediate english",
    "advanced english"
]

# ----------------------------
# 4️⃣ Funciones auxiliares
# ----------------------------
def detect_years(text):
    pattern = r"(\d+)[\+\-]?\d*\s+years?"
    match = re.search(pattern, text.lower())
    return match.group(0) if match else ""

def detect_english(text):
    for level in english_levels:
        if level.lower() in text.lower():
            return level
    return ""

# ----------------------------
# 5️⃣ Descargar datos
# ----------------------------
response = requests.get(API_URL)

if response.status_code != 200:
    raise Exception("Error conectando con Remotive API")

data = response.json()
jobs = data["jobs"]

print(f"🔎 {len(jobs)} vacantes obtenidas")

# ----------------------------
# 6️⃣ Procesar vacantes
# ----------------------------
all_jobs = []

for job in jobs:
    combined_text = f"""
        {job.get('title','')}
        {job.get('description','')}
        {job.get('company_name','')}
        {' '.join(job.get('tags', []))}
    """

    # filtrar por keywords
    if not any(k.lower() in combined_text.lower() for k in KEYWORDS):
        continue

    # experiencia
    level = ""
    for xp in xp_levels:
        if xp in combined_text.lower():
            level = xp
            break

    years = detect_years(combined_text)
    english = detect_english(combined_text)

    all_jobs.append({
        "date": job.get("publication_date"),
        "company": job.get("company_name"),
        "position": job.get("title"),
        "location": job.get("candidate_required_location"),
        "tags": ", ".join(job.get("tags", [])),
        "remote": True,
        "experience_level": level,
        "experience_years": years,
        "english_level": english,
        "url": job.get("url")
    })

# ----------------------------
# 7️⃣ Guardar CSV
# ----------------------------
df = pd.DataFrame(all_jobs)

today = datetime.today().strftime("%Y_%m_%d")
csv_file = os.path.join(RAW_DATA_DIR, f"remotive_jobs_{today}.csv")

df.to_csv(csv_file, index=False)

print(f"✅ {len(df)} vacantes guardadas en:")
print(csv_file)


🔎 25 vacantes obtenidas
✅ 24 vacantes guardadas en:
d:\PROYECTO JOB\Job-Market-Analysis-\data\raw\remotive_jobs_2026_02_09.csv


In [ ]:
Extraer información de Computrabajo

In [48]:
# ----------------------------
# Librerías
# ----------------------------
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time
import re
from datetime import datetime

# ----------------------------
# Carpeta salida
# ----------------------------
BASE_DIR = os.getcwd()
RAW_DATA_DIR = r"D:\PROYECTO JOB\Job-Market-Analysis-\data\raw"
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# ----------------------------
# Configuración
# ----------------------------
BASE_URL = "https://www.computrabajo.com.co"
QUERY = "data engineer"
MAX_PAGES = 3

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# ----------------------------
# Funciones auxiliares
# ----------------------------
def detect_tags(text):
    keywords = [
        "python", "java", "sql", "cloud", "aws", "azure",
        "docker", "kubernetes", "react", "node", "security",
        "support", "software", "backend", "frontend"
    ]

    found = []
    text = text.lower()

    for k in keywords:
        if k in text:
            found.append(k)

    return ", ".join(found)


def detect_experience(text):
    match = re.search(r'(\d+)\+?\s*años', text.lower())
    if match:
        return match.group(1)
    return ""


def detect_english(text):
    if "inglés" in text.lower() or "english" in text.lower():
        return "required"
    return ""


def detect_level(title):
    t = title.lower()

    if "senior" in t:
        return "senior"
    if "junior" in t:
        return "junior"
    if "manager" in t or "lead" in t:
        return "manager"

    return ""


# ----------------------------
# Scraper
# ----------------------------
jobs_data = []

for page in range(1, MAX_PAGES + 1):

    url = f"{BASE_URL}/trabajo-de-{QUERY}?p={page}"
    print("\nPágina:", page)

    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    job_cards = soup.find_all("article")

    if not job_cards:
        break

    for job in job_cards:

        title_tag = job.find("a")
        if not title_tag:
            continue

        position = title_tag.text.strip()
        link = BASE_URL + title_tag["href"]

        spans = job.find_all("span")

        company = spans[0].text.strip() if len(spans) > 0 else ""
        location = spans[1].text.strip() if len(spans) > 1 else ""

        # ---- Entrar al detalle ----
        try:
            detail_resp = requests.get(link, headers=HEADERS)
            detail_soup = BeautifulSoup(detail_resp.text, "html.parser")

            description = detail_soup.get_text(" ", strip=True)

        except:
            description = ""

        tags = detect_tags(description)
        years = detect_experience(description)
        english = detect_english(description)
        level = detect_level(position)

        remote = True if "remoto" in description.lower() else False

        jobs_data.append({
            "date": datetime.utcnow().isoformat(),
            "company": company,
            "position": position,
            "location": location,
            "tags": tags,
            "remote": remote,
            "experience_level": level,
            "experience_years": years,
            "english_level": english,
            "url": link
        })

        print("✔", position)

        time.sleep(1)

    time.sleep(3)

# ----------------------------
# Guardar CSV
# ----------------------------
df = pd.DataFrame(jobs_data)

today = datetime.today().strftime("%Y_%m_%d")
csv_file = os.path.join(
    RAW_DATA_DIR,
    f"computrabajo3_jobs_{today}.csv"
)

df.to_csv(csv_file, index=False)

print("\nVacantes guardadas:", len(df))
print("Archivo:", csv_file)



Página: 1


C:\Users\carlo\AppData\Local\Temp\ipykernel_24340\690060487.py:126: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date": datetime.utcnow().isoformat(),


✔ Ingeniero Datos
✔ Ingeniero de Datos
✔ Data / Analytics Engineer
✔ Data Migration Engineer 48157
✔ Ingeniero de Data 47788
✔ Ingeniero de datos
✔ Analista de Datos
✔ Ingeniero en sistemas o carreras afines de datos 6 años de experiencia en informatica de salud
✔ Ingeniero de Bases de Datos / Database Administrator
✔ Ingeniero de Integración de Datos AWS
✔ Ingeniero de Datos
✔ Ingeniero de datos  Chia
✔ Profesional en ingeniera de datos
✔ Ingeniero de datos
✔ Ingeniero de datos
✔ Practicante profesional en ingeniería de ciencias de datos, ingeniero industrial o carreras afines.
✔ Ingeniero/a de Canales Digitales / Análisis de datos
✔ Consultor Ingeniero de Datos
✔ Ingeniero de confiabilidad y analítica de datos
✔ Data Engineer / Ingeniero(a) de Datos / Especialista en Ingeniería de Datos

Página: 2
✔ Ingeniero/a de analítica de datos
✔ Ingeniero Desarrollador Base de Datos Oracle
✔ Ingeniero experto en Databricks
✔ analista de datos
✔ Software Integration Engineer
✔ Software Integrati

In [ ]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time
from datetime import datetime

# ----------------------------
# Carpeta salida
# ----------------------------
RAW_DATA_DIR = r"D:\PROYECTO JOB\Job-Market-Analysis-\data\raw"
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# ----------------------------
# Configuración
# ----------------------------
QUERY = "software engineer"
LOCATION = "Colombia"
MAX_PAGES = 3

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# ----------------------------
# Scraper
# ----------------------------
jobs_data = []

for page in range(MAX_PAGES):

    start = page * 25

    url = (
        "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
        f"?keywords={QUERY}&location={LOCATION}&start={start}"
    )

    print("\nPágina:", page + 1)

    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    jobs = soup.find_all("li")

    for job in jobs:
        try:
            title = job.find("h3").text.strip()
            company = job.find("h4").text.strip()
            location = job.find("span", class_="job-search-card__location").text.strip()

            link = job.find("a")["href"]

            jobs_data.append({
                "date": datetime.utcnow().isoformat(),
                "company": company,
                "position": title,
                "location": location,
                "tags": QUERY,
                "remote": "",
                "experience_level": "",
                "experience_years": "",
                "english_level": "",
                "url": link
            })

            print("✔", title)

        except:
            continue

    time.sleep(2)

# ----------------------------
# Guardar CSV
# ----------------------------
df = pd.DataFrame(jobs_data)

today = datetime.today().strftime("%Y_%m_%d")
csv_file = os.path.join(
    RAW_DATA_DIR,
    f"linkedin4_jobs_{today}.csv"
)

df.to_csv(csv_file, index=False, encoding="utf-8-sig")

print("\nVacantes guardadas:", len(df))
print("Archivo:", csv_file)



Página: 1


C:\Users\carlo\AppData\Local\Temp\ipykernel_24340\2739875524.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date": datetime.utcnow().isoformat(),


✔ Junior Software Engineer
✔ Software Engineer Frontend
✔ Software Engineer Frontend - IT Mercado Pago
✔ Full Stack Developer Junior
✔ Junior Software Engineer
✔ Software Engineer Backend I
✔ Web Developer
✔ Junior Web Developer
✔ Junior Software Engineer
✔ Software Engineer - Mercado Pago IT

Página: 2
✔ Software Engineer Backend
✔ Software Engineer
✔ Web Developer
✔ Junior Fullstack React & Node Developer - Remote Work | REF#281820
✔ Programador Junior- Data Engineering
✔ Software Engineer Associate
✔ Software Engineer I
✔ Fullstack Software Engineer Associate
✔ Programador Junior- Data Engineering
✔ Aprendiz técnico programación de software

Página: 3
✔ Software Engineer
✔ Fullstack Software Engineer Associate
✔ Front end Software Engineer Associate
✔ Desarrollador/a  MID Fullstack
✔ Software Developer
✔ Desarrollador de Software
✔ Software Engineer Full stack II
✔ Software Engineer Associate
✔ Desarrollador de Software
✔ .NET Back-end

Vacantes guardadas: 30
Archivo: D:\PROYECTO JO